In [1]:
import yfinance as yf
import talib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from datetime import datetime
import pytz

# Define the stock ticker symbol
ticker = 'ONGC.NS'

# Get historical daily data for the stock (1 year)
data = yf.download(ticker, period='1y', interval='1d',threads=True, auto_adjust=True)

# Ensure data is not empty
if data.empty:
    raise ValueError(f"No data found for ticker {ticker}. Please check the ticker and try again.")

# Preserve the original Close values for unscaled reference
data['Close_original'] = data['Close']
close_prices = data['Close'].values.flatten()

# Ensure all arrays are 1D and handle NaN values
high_prices = data['High'].values.flatten()
low_prices = data['Low'].values.flatten()

if np.any(np.isnan(high_prices)) or np.any(np.isnan(low_prices)) or np.any(np.isnan(close_prices)):
    data.ffill(inplace=True)
    high_prices = data['High'].values.flatten()
    low_prices = data['Low'].values.flatten()
    close_prices = data['Close'].values.flatten()

# Calculate technical indicators
data['SMA'] = data['Close'].rolling(window=30).mean()
data['RSI'] = talib.RSI(close_prices, timeperiod=14)
data['MACD'], data['MACD_signal'], _ = talib.MACD(
    close_prices, fastperiod=12, slowperiod=26, signalperiod=9
)
data['ADX'] = talib.ADX(high_prices, low_prices, close_prices, timeperiod=14)

# Bollinger Bands
upper_band, middle_band, lower_band = talib.BBANDS(close_prices, timeperiod=20)
data['Upper_Band'] = upper_band
data['Middle_Band'] = middle_band
data['Lower_Band'] = lower_band

# ATR (Average True Range)
data['ATR'] = talib.ATR(high_prices, low_prices, close_prices, timeperiod=14)

# EMA (Exponential Moving Average)
data['EMA'] = talib.EMA(close_prices, timeperiod=30)

# Handle missing values
data.ffill(inplace=True)
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.ffill(inplace=True)

# Initialize scalers
scaler_close = StandardScaler()
scaler_features = StandardScaler()

# Scale the Close feature separately for later inverse transformation
data['Close_scaled'] = scaler_close.fit_transform(data[['Close']])

# Scale other features for training
scaled_features = scaler_features.fit_transform(
    data[['SMA', 'RSI', 'MACD', 'MACD_signal', 'ADX', 'Upper_Band', 'Middle_Band', 'Lower_Band', 'ATR', 'EMA']]
)
data[['SMA', 'RSI', 'MACD', 'MACD_signal', 'ADX', 'Upper_Band', 'Middle_Band', 'Lower_Band', 'ATR', 'EMA']] = scaled_features

# Define features and target
X = data[['SMA', 'RSI', 'MACD', 'MACD_signal', 'ADX', 'Upper_Band', 'Middle_Band', 'Lower_Band', 'ATR', 'EMA']]
y = data['Close_scaled']

# Train-Test Split (no shuffle due to time-series nature)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Get live minute-level data for predictions
live_data = yf.download(ticker, period='1d', interval='1m', auto_adjust=True)

# Ensure live_data has enough rows
required_rows = max(30, 20, 14)  # Max window size of indicators
if live_data.shape[0] < required_rows:
    raise ValueError("Insufficient live data for technical indicator calculations.")

# Calculate indicators for live data
live_data['SMA'] = live_data['Close'].rolling(window=30).mean()
live_data['RSI'] = talib.RSI(live_data['Close'].to_numpy().flatten(), timeperiod=14)
live_data['MACD'], live_data['MACD_signal'], _ = talib.MACD(
    live_data['Close'].to_numpy().flatten(), fastperiod=12, slowperiod=26, signalperiod=9
)
live_data['ADX'] = talib.ADX(
    live_data['High'].to_numpy().flatten(),
    live_data['Low'].to_numpy().flatten(),
    live_data['Close'].to_numpy().flatten(),
    timeperiod=14
)
upper_band, middle_band, lower_band = talib.BBANDS(live_data['Close'].to_numpy().flatten(), timeperiod=20)
live_data['Upper_Band'] = upper_band
live_data['Middle_Band'] = middle_band
live_data['Lower_Band'] = lower_band
live_data['ATR'] = talib.ATR(
    live_data['High'].to_numpy().flatten(), live_data['Low'].to_numpy().flatten(), live_data['Close'].to_numpy().flatten(), timeperiod=14
)
live_data['EMA'] = talib.EMA(live_data['Close'].to_numpy().flatten(), timeperiod=30)

# Handle missing values and scale features
live_data.ffill(inplace=True)
live_data_scaled = scaler_features.transform(
    live_data[['SMA', 'RSI', 'MACD', 'MACD_signal', 'ADX', 'Upper_Band', 'Middle_Band', 'Lower_Band', 'ATR', 'EMA']]
)

# Predict on live data
predictions = model.predict(live_data_scaled)

# Extract the predicted closing price for the most recent minute
predicted_closing_price_scaled = predictions[-1]
predicted_closing_price = scaler_close.inverse_transform([[predicted_closing_price_scaled]])[0][0]

# Get the most recent minute's data
latest_time = live_data.index[-1]

# Convert latest_time to local timezone (India)
india_tz = pytz.timezone('Asia/Kolkata')

# Check if latest_time is naive or already timezone-aware
if latest_time.tzinfo is None:
    # If naive, localize to UTC first, then convert
    local_time = latest_time.tz_localize('UTC').tz_convert(india_tz)
else:
    # If already aware, directly convert
    local_time = latest_time.tz_convert(india_tz)


# Get the previous closing price from the original data
previous_closing_price = data['Close_original'].iloc[-1]

# Check if the model predicted a gain or loss
if predicted_closing_price > previous_closing_price:
    status = 'Gain'
    price_difference = predicted_closing_price - previous_closing_price
    gain_loss_percentage = (price_difference / previous_closing_price) * 100
else:
    status = 'Loss'
    price_difference = previous_closing_price - predicted_closing_price
    gain_loss_percentage = (price_difference / previous_closing_price) * 100

# Print results
print(f"Latest Closing Price: {previous_closing_price:.2f}")
print(f"Predicted Closing Price for {local_time} (IST): {predicted_closing_price:.2f}")
print(f"Price Difference: {price_difference:.2f} ({gain_loss_percentage:.2f}%)")
print(f"Prediction Status: {status}")


[*********************100%***********************]  1 of 1 completed


Mean Squared Error: 3.081808547623752


[*********************100%***********************]  1 of 1 completed

Latest Closing Price: 281.85
Predicted Closing Price for 2026-04-06 15:29:00+05:30 (IST): 235.97
Price Difference: 45.88 (16.28%)
Prediction Status: Loss
